# Agent

Agents combine language <b>models</b> with <b>tools</b> to create systems that can reason about tasks, decide which tools to use, and iteratively work towards solutions.

<b>create_agent</b> provides a production-ready agent implementation

An LLM Agent runs tools in a loop to achieve a goal. An agent runs until a stop condition is met - i.e., when the model emits a final output or an iteration limit is reached.

<img src="agent_flow.jpg" alt="Step 1 is the analyze calling tool in this graph">

create_agent builds a <b>graph-based agent</b> runtime using <b>LangGraph</b>. A graph consists of nodes (steps) and edges (connections) that define how your agent processes information. The agent moves through this graph, executing nodes like the model node (which calls the model), the tools node (which executes tools), or middleware.

# Core components

## Models

The model is the reasoning engine of your agent. It can be specified in multiple ways, supporting both static and dynamic model selection.

### Static model

Static models are configured once when creating the agent and remain unchanged throughout execution. This is the most common and straightforward approach.

In [ ]:
# Too simple
from langchain.agents import create_agent

agent = create_agent(
    "gemini-2.5-flash",
    tools=tools
)

ImportError: cannot import name 'create_agent' from 'langchain.agents' (c:\Users\Learner\.conda\envs\langchain-book\Lib\site-packages\langchain\agents\__init__.py)

In [ ]:
# More control option
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-5",
    temperature=0.1,
    max_tokens=1000,
    timeout=30
    # ... (other params)
)
agent = create_agent(model, tools=tools)

### Dynamic model

Dynamic models are selected at runtime based on the current state and context. This enables sophisticated routing logic and cost optimization.

To use a dynamic model, create middleware using the @wrap_model_call decorator that modifies the model in the request:

In [ ]:
ัfrom langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse


basic_model = ChatOpenAI(model="gpt-4o-mini")
advanced_model = ChatOpenAI(model="gpt-4o")

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """Choose model based on conversation complexity."""
    message_count = len(request.state["messages"])

    if message_count > 10:
        # Use an advanced model for longer conversations
        model = advanced_model
    else:
        model = basic_model

    return handler(request.override(model=model))

agent = create_agent(
    model=basic_model,  # Default model
    tools=tools,
    middleware=[dynamic_model_selection]
)

# Tools

Tools give agents the ability to take actions. Agents go beyond simple model-only tool binding by facilitating:
- Multiple tool calls in sequence (triggered by a single prompt)
- Parallel tool calls when appropriate
- Dynamic tool selection based on previous results
- Tool retry logic and error handling
- State persistence across tool calls


For more information, see Tools.

Tools can be specified as plain Python functions or coroutines.

The tool decorator can be used to customize tool names, descriptions, argument schemas, and other properties.

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent


@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

agent = create_agent(model, tools=[search, get_weather])

If an empty tool list is provided, the agent will consist of a single LLM node without tool-calling capabilities.

## Tool error handling

To customize how tool errors are handled, use the @wrap_tool_call decorator to create middleware:

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage


@wrap_tool_call
def handle_tool_errors(request, handler):
    """Handle tool execution errors with custom messages."""
    try:
        return handler(request)
    except Exception as e:
        # Return a custom error message to the model
        return ToolMessage(
            content=f"Tool error: Please check your input and try again. ({str(e)})",
            tool_call_id=request.tool_call["id"]
        )

agent = create_agent(
    model="gpt-4o",
    tools=[search, get_weather],
    middleware=[handle_tool_errors]
)

The agent will return a ToolMessage with the custom error message when a tool fails:

In [ ]:
# result of error tool call
[
    ...
    ToolMessage(
        content="Tool error: Please check your input and try again. (division by zero)",
        tool_call_id="..."
    ),
    ...
]

SyntaxError: invalid syntax. Perhaps you forgot a comma? (512646947.py, line 2)

# Tool use in the ReAct loop

Agents follow the ReAct (“Reasoning + Acting”) pattern, alternating between brief reasoning steps with targeted tool calls and feeding the resulting observations into subsequent decisions until they can deliver a final answer.

Just for Example

Prompt: Identify the current most popular wireless headphones and verify availability.

================================ Human Message =================================

Find the most popular wireless headphones right now and check if they're in stock

Reasoning: “Popularity is time-sensitive, I need to use the provided search tool.”

Acting: Call search_products("wireless headphones")

================================== Ai Message ==================================
Tool Calls:
  search_products (call_abc123)
 Call ID: call_abc123
  Args:
    query: wireless headphones

================================= Tool Message =================================

Found 5 products matching "wireless headphones". Top 5 results: WH-1000XM5, ...

Reasoning: “I need to confirm availability for the top-ranked item before answering.”
Acting: Call check_inventory("WH-1000XM5")

================================== Ai Message ==================================
Tool Calls:
  check_inventory (call_def456)
 Call ID: call_def456
  Args:
    product_id: WH-1000XM5

================================= Tool Message =================================

Product WH-1000XM5: 10 units in stock

Reasoning: “I have the most popular model and its stock status. I can now answer the user’s question.”

Acting: Produce final answer

================================== Ai Message ==================================

I found wireless headphones (model WH-1000XM5) with 10 units in stock...

# System prompt

You can shape how your agent approaches tasks by providing a prompt. The system_prompt parameter can be provided as a string:

In [2]:
agent = create_agent(
    model,
    tools,
    system_prompt="You are a helpful assistant. Be concise and accurate."
)

NameError: name 'create_agent' is not defined

## Dynamic system prompt

For more advanced use cases where you need to modify the system prompt based on runtime context or agent state, you can use middleware.
The @dynamic_prompt decorator creates middleware that generates system prompts based on the model request:

In [ ]:
from typing import TypedDict

from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        return f"{base_prompt} Explain concepts simply and avoid jargon."

    return base_prompt

agent = create_agent(
    model="gpt-4o",
    tools=[web_search],
    middleware=[user_role_prompt],
    context_schema=Context
)

# The system prompt will be set dynamically based on context
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain machine learning"}]},
    context={"user_role": "expert"}
)

# Invocation

You can invoke an agent by passing an update to its State. All agents include a sequence of messages in their state; to invoke the agent, pass a new message:

In [1]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)

NameError: name 'agent' is not defined

For streaming steps and / or tokens from the agent, refer to the streaming guide.


Otherwise, the agent follows the LangGraph Graph API and supports all associated methods, such as stream and invoke.